- an ensemble is a model that contains a number of base models predicting together
- the base models can all be the same or different
- the ensemble models we covered have had the copies of the same base model - a decision tree:
    - bagged trees and random forest
    - all boosting models
- in general, we can bring a number of any models together -- not necessary copies of the same base model


## Voting Ensemble -- Regression

- a voting ensemble model trains a number of base models on the same training data and makes predictions by using a simple aggregation of all its trained base models
- the base models are different, not copies of one another
- for regression, the ensemble model makes predictions by averaging the predictions of all base models

# Voting Ensemble -- Classification

- two options for ensemble prediction of classification:
    - hard voting (mode prediction)
    - soft (probabilistic) voting: average of the prediction probabilities and then obtaining a class prediction

# Should all base models be equally important?

- in bagging all base models were equally important
- in boosting, they were not
- what to do with the different base models of a voting ensemble?

If not, can we assign them different weights?

How would we know what the weights of the base models should be?

- we cannot arbitrarily assign weights to the models and hope for the best
- trying different weight combos and cross-validating would be incredibly costly
- or we can just let another model take care of finding the weights, bringing us to Stacking Ensemble

# Stacking Ensemble

- trains a number of base models on the same training data
- gets their predictions
- uses these predictions as predictors to train a final model
    - called the "meta" model
    - a simpler model is ussually preferred, such as Linear/Logistic Regression, Lasso, or a single decision tree
    - the true resposne values would be the same for the final model

# Training and Tuning Ensemble Models

- first option:
    - first train and tune each base model separately
    - then bring them together for aggregation or meta model
    - not guaranteed to find best combination
- second option:
    - train and tune all hyperparameters of all the different models simultaneously
    - for stacking, includes hyperparameters of the meta model as well
    - guarantees best hyperparameter combo
    - VERY computationally intensive

Do all base models see the entire training data or a subset of it?
- Since the base models are different, they are already fitting to the data in different ways, so using subsets of the training data is more likely to increase bias than to reduce variance
- Scikit-learn does not give the option of subsets; manual implementation would be
necessary

# Intuition

- for voting and stacking ensembles, we assume that using different models (trained in differen ways) together is expected to improve the generalizability and robustness of the model 
    - some base models will return a good prediction and make up for others that don't
    - if some model(s) are affected by outliers, the ones that are not affected will make up for them

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# More libraries/tools here
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier

from sklearn.ensemble import VotingRegressor, VotingClassifier, StackingRegressor, StackingClassifier

In [2]:
# Training data
trainf = pd.read_csv('/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/Car_features_train.csv') # Predictors
trainp = pd.read_csv('/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/Car_prices_train.csv') # Response
train = pd.merge(trainf,trainp)
train.head()

# Test data
testf = pd.read_csv('/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/Car_features_test.csv') # Predictors
testp = pd.read_csv('/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/Car_prices_test.csv') # Response
test = pd.merge(testf,testp)
test.head()

predictors = ['mpg', 'engineSize', 'year', 'mileage']

X_train = train[predictors]
y_train = train['price']

X_test = test[predictors]
y_test = test['price']

depending on the base models, we may want to scale the data

## Regression

In [3]:
bm1 = RandomForestRegressor(random_state=12, n_estimators=200, max_features=0.9, max_samples=0.9, bootstrap=True)

bm2 = KNeighborsRegressor(n_neighbors=15, weights = 'distance')

bm3 = XGBRegressor(random_state=12, max_depth=5, n_estimators=100, learning_rate=0.1, reg_lambda=0.1, gamma=0, subsample=0.9)

In [4]:
model = VotingRegressor(
    estimators = [
        ("rf", bm1),
        ("knn", bm2),
        ("xgb", bm3)
    ]
)

model.fit(X_train, y_train)

VotingRegressor(estimators=[('rf',
                             RandomForestRegressor(max_features=0.9,
                                                   max_samples=0.9,
                                                   n_estimators=200,
                                                   random_state=12)),
                            ('knn',
                             KNeighborsRegressor(n_neighbors=15,
                                                 weights='distance')),
                            ('xgb',
                             XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_round...
                                          feature_weights=None, gamma=0,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=0.1, max_bin=None,
                                          max_cat_threshold=None,
                                          max_cat_to_onehot=None,
                                          max_delta_step=None, max_depth=5,
                                          max_leaves=None,
                                          min_child_weight=None, missing=nan,
                                          monotone_constraints=None,
                                          multi_strategy=None, n_estimators=100,
                                          n_jobs=None, num_parallel_tree=None, ...))])

In [ ]:
# to tune the ensemble model, we can tune all base models separately and then bring them together, or 

# create the model
model = VotingRegressor(
    estimators=[
        ("rf", bm1),
        ("knn", bm2),
        ("xgb", bm3)
    ],
)

# create the grid
grid = {
    "rf__max_samples": [...],
    "rf__max_features": [...],
    # same with knn and xgb hyper parameters -- very expensive
}

In [6]:
# a lower level way to implement voting ensembles

# assuming the base models are already tuned -- train them
bm1.fit(X_train, y_train)
bm2.fit(X_train, y_train)
bm3.fit(X_train, y_train)

# average the predictions for the test data
(bm1.predict(X_test) + bm2.predict(X_test) + bm3.predict(X_test)) / 3

array([37983.359376  , 22745.41512537, 42874.11138256, ...,
       22339.53211662, 18439.95984486,  4513.9050973 ], shape=(2672,))

# Stacking Ensemble

In [7]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression

cv_settings_meta = KFold(n_splits = 4, random_state = 1, shuffle = True)
model = StackingRegressor(
    estimators=[
        ("rf", bm1),
        ("knn", bm2),
        ("xgb", bm3)
    ],
    final_estimator = LinearRegression(),
    cv = cv_settings_meta # cv settings that determine the cv predictions of the base models
)

- if the meta model is linear regression, there is nothing left to tune after base models are tuned
- for lasso/ridge/decision tree, the meta model has hyperparameters to tune

In [ ]:
# first way to tune

# create the StackingRegressor as above, in the grid, only use the meta model hyperparameters
from sklearn.linear_model import Lasso

cv_settings_meta = KFold(n_splits = 4, random_state = 1, shuffle = True)
model = StackingRegressor(
    estimators=[
        ("rf", bm1),
        ("knn", bm2),
        ("xgb", bm3)
    ],
    final_estimator = Lasso(),
    cv = cv_settings_meta # cv settings that determine the cv predictions of the base models
)

grid = {
    'final_estimator__alpha': [0.01, 0.1, 1, 10]
}

In [10]:
from sklearn.model_selection import cross_val_predict

# second way to tune: the lower-level implementation

pred_cv1 = cross_val_predict(bm1, X_train, y_train, cv = cv_settings_meta)
pred_cv2 = cross_val_predict(bm2, X_train, y_train, cv = cv_settings_meta)
pred_cv3 = cross_val_predict(bm3, X_train, y_train, cv = cv_settings_meta)

# Stack the pred_cvs into one dataframe, train/tune Lasso,Ridge, Decision Tree etc. as the meta model


## Classification

In [4]:
bm1 = RandomForestClassifier(random_state=12, n_estimators=200, max_features=0.9, max_samples=0.9, bootstrap=True)

bm2 = KNeighborsClassifier(n_neighbors=15, weights = 'distance')

bm3 = XGBClassifier(random_state=12, max_depth=5, n_estimators=100, learning_rate=0.1, reg_lambda=0.1, gamma=0, subsample=0.9)

# Voting Ensemble

In [ ]:
model = VotingClassifier(
    estimators=[
        ("rf", bm1),
        ("knn", bm2),
        ("xgb", bm3)
    ],
    voting='soft'  # 'hard' for majority voting, 'soft' for averaging predicted probabilities
)

In [ ]:
# If you implement a VotingClassifier the lower-level way, you can use tuned thresholds as well!


bm1.fit(X_train, y_train)
bm2.fit(X_train, y_train)
bm3.fit(X_train, y_train)

(bm1.predict_proba(X_test)[:,1] > thr1 + 
 bm2.predict_proba(X_test)[:,1] > thr2 + 
 bm3.predict_proba(X_test)[:,1] > thr3)/3 # Round this value for majority voting

# Stacking Ensemble

In [11]:
from sklearn.linear_model import LogisticRegression

model = StackingClassifier(
    estimators = [('rf', bm1), ('knn', bm2), ('xgb', bm3)], # The estimators input is identical to Voting Ensemble
    final_estimator = LogisticRegression(penalty = 'l2'), # This is the meta model.
    cv = cv_settings_meta,
    stack_method = 'predict' # for hard voting or 'predict_proba' for soft voting
)

# You can use the lower-level approach as well with cross_val_predict, just use method = "predict_proba" for
# soft voting